# Lab 2 - Data Exploration

**Objective:** run analytical SQL against the Iceberg table you populated in Lab 1, and chart the results with `matplotlib`.

**Estimated time:** ~20 minutes.

**You will:**

1. Connect to Trino with both the native `trino-python-client` driver and with `SQLAlchemy` (choose whichever fits your workflow).
2. Execute four analytical queries and visualize the results inline.

**Prerequisites:** Lab 1 completed successfully (`iceberg.airline_lab.flights` is populated).

**Verification:** at least one `matplotlib` chart renders inline at the end of the notebook.

## Option A - `trino-python-client` (native DB-API)

Use this driver when you want fine-grained control over sessions, streaming result sets, or non-default authentication. This lab wraps it in a helper function `q(sql)` that returns a pandas DataFrame.

In [ ]:
import os
import pandas as pd
from trino.dbapi import connect
from trino.auth import BasicAuthentication

conn = connect(
    host=os.environ['TRINO_HOST'],
    port=int(os.environ.get('TRINO_PORT', 443)),
    user=os.environ['TRINO_USER'],
    auth=BasicAuthentication(os.environ['TRINO_USER'], os.environ['TRINO_PASSWORD']),
    http_scheme='https',
    catalog='iceberg',
    schema='airline_lab',
)

USERNAME = os.environ['TRINO_USER']

def q(sql):
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(rows, columns=cols)

q(f'SELECT COUNT(*) AS total FROM flights_{USERNAME}')

## Option B - SQLAlchemy (`sqlalchemy-trino`)

Use this path for pandas (`pd.read_sql`), for BI tools, or for any ORM that speaks SQLAlchemy. The connection string carries the same credentials as Option A.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

user = quote_plus(os.environ['TRINO_USER'])
pw   = quote_plus(os.environ['TRINO_PASSWORD'])
host = os.environ['TRINO_HOST']
port = int(os.environ.get('TRINO_PORT', 443))

engine = create_engine(
    f'trino://{user}:{pw}@{host}:{port}/iceberg/airline_lab',
    connect_args={'http_scheme': 'https'},
)

USERNAME = os.environ['TRINO_USER']

pd.read_sql(f'SELECT COUNT(*) AS total FROM flights_{USERNAME}', engine)

## Query 1 - Top 10 departure countries

A simple `GROUP BY`/`ORDER BY` to establish the highest-volume departure countries in the dataset.

In [ ]:
USERNAME = os.environ['TRINO_USER']

top_countries = q(f'''
    SELECT country_name, COUNT(*) AS flights
    FROM iceberg.airline_lab.flights_{USERNAME}
    GROUP BY country_name
    ORDER BY flights DESC
    LIMIT 10
''')
top_countries

In [ ]:
%matplotlib inline
top_countries.plot(kind='barh', x='country_name', y='flights', legend=False, figsize=(8,4), title='Top 10 departure countries')

## Query 2 - Flight-status distribution

Counts of `On Time`, `Delayed`, and `Cancelled` across the entire dataset.

In [ ]:
q(f'''
    SELECT flight_status, COUNT(*) AS flights
    FROM iceberg.airline_lab.flights_{USERNAME}
    GROUP BY flight_status
    ORDER BY flights DESC
''')

## Query 3 - Age demographics by continent

Uses `APPROX_PERCENTILE` to compute the 25th, 50th, and 75th percentiles in a single pass. `APPROX_PERCENTILE` is significantly faster than exact percentiles on large datasets.

In [ ]:
q(f'''
    SELECT airport_continent,
           APPROX_PERCENTILE(age, 0.25) AS p25,
           APPROX_PERCENTILE(age, 0.50) AS p50,
           APPROX_PERCENTILE(age, 0.75) AS p75,
           AVG(age)                     AS avg_age
    FROM iceberg.airline_lab.flights_{USERNAME}
    GROUP BY airport_continent
    ORDER BY avg_age DESC
''')

## Query 4 - Monthly departure trend

Bucket departures by month with `date_trunc('month', ...)` and chart the result.

In [ ]:
monthly = q(f'''
    SELECT date_trunc('month', departure_date) AS month,
           COUNT(*) AS flights
    FROM iceberg.airline_lab.flights_{USERNAME}
    GROUP BY date_trunc('month', departure_date)
    ORDER BY month
''')
monthly.plot(x='month', y='flights', figsize=(9, 4), title='Monthly departures')

## Verify

You should see at least one non-empty DataFrame and one chart. If the DataFrames are empty, rerun Lab 1.

**Next:** open `03_advanced_queries.ipynb`.